# GEO-INTEL hosted data run

Run in a Python 3.11 Colab or Kaggle runtime. In Colab, upload a ZIP of the repository when prompted, or set `REPO_URL` to its pushed HTTPS URL. The run installs pinned geospatial dependencies, prints actual runtime/provider/scene information, runs the 5×5 km smoke test and full T1/T2 composites, and saves COGs, metadata, manifest, report, and figures to Drive (Colab) or Kaggle working storage. It does not execute rasterio on Windows.

In [ ]:
REPO_URL = ''  # optional: 'https://github.com/ORG/geo-intel.git'
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
if sys.version_info[:2] != (3, 11): raise RuntimeError(f'Python 3.11 required, got {sys.version}')
if REPO_URL:
    repo = Path('/content/geo-intel') if Path('/content').exists() else Path('/kaggle/working/geo-intel')
    if repo.exists(): shutil.rmtree(repo)
    subprocess.run(['git','clone',REPO_URL,str(repo)], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    zips = [f for f in uploaded if f.lower().endswith('.zip')]
    if len(zips) != 1: raise RuntimeError('Upload exactly one repository ZIP.')
    root = Path('/content/geo-intel'); root.mkdir(exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z: z.extractall(root)
    candidates = [p for p in root.iterdir() if p.is_dir() and (p/'pyproject.toml').exists()]
    repo = candidates[0] if candidates else root
if not (repo/'pyproject.toml').exists(): raise FileNotFoundError('Repository root must contain pyproject.toml')
os.chdir(repo)
print('Repository:', repo.resolve(), '| Python:', sys.version)

In [ ]:
%pip install --quiet rasterio==1.4.3 rioxarray==0.17.0 xarray==2024.6.0 geopandas==1.0.1 shapely==2.0.6 pyproj==3.6.1 pyogrio==0.10.0 pystac-client==0.8.3 stackstac==0.5.1 planetary-computer==1.0.0 dask[array]==2024.8.0 numpy==1.26.4 pandas==2.2.2 scipy==1.14.1 pyyaml==6.0.2 click==8.1.7 rio-cogeo==5.3.3 matplotlib==3.9.2 scikit-learn==1.5.1 pytest==8.3.2 pytest-asyncio==0.23.8 psycopg2-binary==2.9.9
%pip install --quiet --no-deps -e .
import rasterio
from osgeo import gdal
print('Python:',sys.version,'\nGDAL:',gdal.VersionInfo('--version'),'\nrasterio:',rasterio.__version__)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    output = Path('/content/drive/MyDrive/geo_intel_run')
except ImportError:
    output = Path('/kaggle/working/geo_intel_run')
output.mkdir(parents=True,exist_ok=True)

## Full test suite (including raster tests)

This runs plain `pytest` with no marker exclusion, so every test marked `raster` is attempted in the hosted GDAL/Rasterio runtime. The cell saves the complete output and fails if any test fails.

In [ ]:
test_run = subprocess.run([sys.executable,'-m','pytest','-v','--tb=short'],cwd=repo,text=True,capture_output=True)
test_output = test_run.stdout + '\n' + test_run.stderr
print(test_output)
(output/'full_pytest_output.txt').write_text(test_output,encoding='utf-8')
if test_run.returncode != 0: raise RuntimeError(f'Full pytest suite failed with exit code {test_run.returncode}; see full_pytest_output.txt')
print('FULL TEST SUITE PASSED; complete output saved to', output/'full_pytest_output.txt')

In [ ]:
subprocess.run([sys.executable,'scripts/colab_run.py','--repo',str(repo),'--output',str(output),'--provider','aws'],check=True)

## Phase 4 candidate points (no labels created)

Optionally generate QGIS candidate points from the two composites just created. This creates stratified rule-map points and an empty label CSV schema only; candidate classes are not reference labels.

In [ ]:
subprocess.run([sys.executable,'scripts/phase4_candidates.py','--config',str(repo/'config/config.yaml'),'--t1-composite',str(output/'cogs/t1_composite.tif'),'--t2-composite',str(output/'cogs/t2_composite.tif'),'--output-dir',str(output/'phase4_candidates')],cwd=repo,check=True)
print('No reference labels or accuracy results were created.')

## T1 vs T2 previews

The runner creates true-colour RGB and false-colour side-by-side comparisons, plus a valid-pixel map, from the run's COGs.

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(output/'t1_t2_previews.png')))
display(Image(filename=str(output/'valid_pixel_map.png')))
print((output/'data_report.md').read_text())